# Jour 2 · Guider le modèle sans compliquer le prompt


## Commencer par la consigne la plus simple

**Zero-shot** signifie que le modèle reçoit une consigne mais aucun exemple. C'est le bon point de départ : si une instruction claire suffit, ajouter beaucoup de texte rend le prompt plus difficile à maintenir.


![Quatre niveaux de guidage d'un prompt](../assets/jour_01/08_niveaux_prompting.png)

*On ajoute étapes, exemples ou schéma en réponse à un échec observé, pas par réflexe.*


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


prompt_zero_shot = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu aides un technicien helpdesk. Réponds de façon concise et n'invente aucun fait.",
    ),
    ("human", "Analyse ce message : {message}"),
])
chaine_zero_shot = prompt_zero_shot | llm | StrOutputParser()

message = "Mon écran est allumé mais n'affiche plus l'image de mon portable."
print(chaine_zero_shot.invoke({"message": message}))


## Nommer la forme attendue

« Réfléchis étape par étape » demande un processus vague et souvent verbeux. Pour un support opérationnel, nous préférons nommer les éléments utiles à afficher : diagnostic probable, vérification et action.

Le modèle produit ainsi une justification courte que l'utilisateur peut contrôler, sans exposer ni dépendre d'une pensée privée détaillée.


In [ ]:
prompt_etapes = ChatPromptTemplate.from_messages([
    (
        "system",
        "Réponds avec exactement trois rubriques courtes : "
        "Diagnostic, Vérification, Action. "
        "N'ajoute aucun fait absent du message.",
    ),
    ("human", "{message}"),
])
chaine_etapes = prompt_etapes | llm | StrOutputParser()

print(chaine_etapes.invoke({"message": message}))


## Few-shot : montrer un exemple

**Few-shot** signifie fournir quelques couples entrée/sortie. L'exemple montre le niveau de détail et la forme souhaités.

Un exemple n'est pas une règle métier cachée : les règles importantes doivent rester écrites explicitement dans la consigne ou dans du code.


In [ ]:
exemples = [
    {
        "entree": "L'imprimante affiche bourrage papier.",
        "sortie": (
            "Diagnostic : papier probablement bloqué.\n"
            "Vérification : localiser le compartiment indiqué.\n"
            "Action : annuler l'impression puis éteindre l'appareil avant ouverture."
        ),
    },
    {
        "entree": "GlobalProtect indique portail inaccessible.",
        "sortie": (
            "Diagnostic : portail VPN non joignable.\n"
            "Vérification : tester d'abord la connexion Internet.\n"
            "Action : contrôler ensuite l'adresse exacte du portail."
        ),
    },
]


def formater_exemples(items):
    return "\n\n".join(
        f"ENTRÉE : {item['entree']}\nSORTIE :\n{item['sortie']}"
        for item in items
    )


prompt_few_shot = ChatPromptTemplate.from_messages([
    (
        "system",
        "Reproduis la forme des exemples. Reste concis et n'invente aucun fait.\n\n"
        "EXEMPLES :\n{exemples}",
    ),
    ("human", "ENTRÉE : {message}\nSORTIE :"),
])
chaine_few_shot = prompt_few_shot | llm | StrOutputParser()

print(chaine_few_shot.invoke({
    "exemples": formater_exemples(exemples),
    "message": message,
}))


### À vous de jouer — compléter les exemples

Ajoutez un exemple sur une demande de code MFA. La sortie doit montrer un refus clair, sans inventer d'incident supplémentaire.


In [ ]:
# À vous de jouer — ajoutez un exemple entrée/sortie sur la sécurité.
nouvel_exemple = None
if nouvel_exemple is not None:
    exemples.append(nouvel_exemple)
print("Exemples disponibles :", len(exemples))


## Adapter la présentation, pas les faits

Une même information peut être présentée à un débutant, un technicien ou un manager. Les mots et le niveau de détail changent ; les faits restent identiques.


![Un noyau de faits décliné pour trois audiences](../assets/jour_01/08_adapter_audience.png)

*L'adaptation du ton ne donne pas le droit d'ajouter des faits, des coûts ou des délais absents.*


In [ ]:
from pydantic import BaseModel, Field


class VersionsAudience(BaseModel):
    debutant: str = Field(description="Mots simples, une analogie et des étapes explicites.")
    technicien: str = Field(description="Vocabulaire technique précis et vérifications concrètes.")
    manager: str = Field(description="Impact, risque et prochaine décision sans jargon inutile.")


adapter_audiences = llm.with_structured_output(
    VersionsAudience,
    method="json_schema",
)

faits_valides = (
    "Incident VPN confirmé depuis 08:15. Environ 400 utilisateurs sont impactés. "
    "L'astreinte Infrastructure est prévenue. Aucun délai de résolution n'est connu."
)
versions = adapter_audiences.invoke(
    "Réécris uniquement ces faits pour les trois audiences du schéma :\n" + faits_valides
)

print("DÉBUTANT\n", versions.debutant)
print("\nTECHNICIEN\n", versions.technicien)
print("\nMANAGER\n", versions.manager)


## Meta-prompting : utile, mais à contrôler

Le **meta-prompting** demande à un modèle de produire ou d'adapter des prompts. Il peut aider à explorer plusieurs formulations.

Pour une application stable, les prompts retenus doivent toutefois être relus, testés, versionnés et conservés dans le code. Générer un nouveau prompt à chaque requête rend le comportement difficile à reproduire.


## À retenir

- Commencer par une consigne zero-shot claire.
- Nommer les rubriques utiles plutôt que demander un long raisonnement.
- Ajouter quelques exemples lorsque la forme reste instable.
- Utiliser un schéma lorsqu'un programme consomme le résultat.
- Adapter le ton sans modifier les faits.
